<a href="https://colab.research.google.com/github/jach36/Scallop-Population-CV-Project/blob/main/part1_oceanicAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **-- BEFORE RUNNING --**
### Ensure that the CV-Final-Project and CV-ObjectDetection folders are in your MyDrive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# install dependencies
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U ultralytics

In [ ]:
# Create a local directory
!mkdir -p /content/scallop_data

# Copy everything from shared drive to your local Colab disk (so we have write permissions)
!cp -r -v /content/drive/MyDrive/CV-ObjectDetection/object_detection_dataset/* /content/scallop_data/

In [ ]:
import yaml

local_yaml = {
    'path': '/content/scallop_data',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 7,
    'names': ['crab', 'eel', 'flatfish', 'roundfish', 'scallop', 'skate', 'whelk']
}

with open('/content/local_data.yaml', 'w') as f:
    yaml.dump(local_yaml, f)

In [ ]:
import cv2
import os
from tqdm import tqdm

# Define paths
drive_path = "/content/drive/MyDrive/CV-Final-Project/dataset"
local_cropped = "/content/single_frame_dataset"
years = ["2016", "2018", "2019", "2022", "2024"]

for year in years:
    os.makedirs(f"{local_cropped}/{year}", exist_ok=True)
    in_dir = os.path.join(drive_path, year)
    print(f"Cropping {year}...")

    images = [f for f in os.listdir(in_dir) if f.endswith(('.png', '.jpg'))]
    for name in tqdm(images):
        img = cv2.imread(os.path.join(in_dir, name))
        if img is not None:
            # Crop to the LEFT frame (0 to 1360 pixels) (some images require cropping)
            cv2.imwrite(os.path.join(local_cropped, year, name), img[:, :1360])

In [ ]:
from ultralytics import YOLO

# Load the pretrained model
model = YOLO('/content/yolov8s.pt')

# Start training using the YAML inside the scallop_data folder
results = model.train(
    data='/content/scallop_data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='scallop_detection'
)

In [ ]:
# Create a folder in your drive for your models
!mkdir -p /content/drive/MyDrive/models/

# Copy over the weights (may need to change paths depending on MyDrive format)
!cp /content/runs/detect/scallop_detection/part1_detection/weights/best.pt /content/drive/MyDrive/models/scallop_weights.pt
print("Model saved to drive.")

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

# replace with path of model
model_path = '/content/runs/detect/scallop_detection/part1_detection/weights/best.pt'
if not os.path.exists(model_path):
    print("Warning: Local path lost. Loading from Google Drive instead...")
    model_path = '/content/drive/MyDrive/models/scallop_v3.pt'

model = YOLO(model_path)

drive_path = "/content/drive/MyDrive/CV-Final-Project/dataset"
years = ["2016", "2018", "2019", "2022", "2024"]

results_list = []

for year in years:
    folder_path = os.path.join(drive_path, year)
    if not os.path.exists(folder_path):
        print(f"Skipping {year}, folder not found.")
        continue

    print(f"Analyzing {year}...")

    # Use stream=True to prevent RAM crashes (deletes from RAM after detection)
    results_generator = model.predict(
        source=folder_path,
        save=True,
        save_txt=True,
        conf=0.3,
        project="Population_Analysis",
        name=year,
        stream=True
    )

    total_found = 0
    img_count = 0

    # Iterate through the generator one by one
    for r in results_generator:
        total_found += len(r.boxes)
        img_count += 1

    results_list.append({
        "Year": year,
        "Total Images": img_count,
        "Total Scallops": total_found,
        "Scallops Per Image": round(total_found / img_count, 2) if img_count > 0 else 0
    })

summary_df = pd.DataFrame(results_list)
print("\n--- Scallop Population Summary Table ---")
print(summary_df)
summary_df.to_csv("scallop_population_results.csv", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("scallop_population_results.csv")

plt.figure(figsize=(10, 6))
sns.lineplot(x=df["Year"], y=df["Scallops Per Image"], marker='o', linewidth=2.5, color='#2c7bb6')
plt.fill_between(df["Year"], df["Scallops Per Image"], alpha=0.2, color='#2c7bb6')

plt.title('Scallop Population Trend (2015 - 2024)', fontsize=16)
plt.xlabel('Survey Year', fontsize=12)
plt.ylabel('Average Scallops per Image', fontsize=12)
plt.xticks(df["Year"])
plt.grid(True, linestyle='--', alpha=0.6)
plt.ylim(0, df["Scallops Per Image"].max() + 0.5)

plt.savefig('population_trend.png', dpi=300)
plt.show()